# Lab 06-03: Inference Tables and Monitoring

**Module:** 06 — Evaluation & Monitoring (12% of exam)  
**UI Sections:** Serving | SQL Editor | Catalog  
**Estimated Time:** 45–55 minutes  
**Difficulty:** Intermediate

---

## What & Why

Inference tables automatically log every request/response from serving endpoints. The exam
tests understanding what is logged, where it is stored, and how to use it for monitoring,
debugging, and evaluation. This is the bridge between offline evaluation (Lab 06-01) and
production monitoring -- once your model is serving traffic, inference tables capture
everything you need to assess quality in the real world.

---

## Mental Model

> **Analogy:** Inference tables are like a flight data recorder (black box) for your AI.
> Every conversation -- input, output, latency, tokens used -- is automatically recorded
> in a Delta table. When something goes wrong (bad answer, slow response), you have the
> full flight log to investigate.

---

## Exam Alert

| # | Trap | Correct Answer |
|---|------|----------------|
| 1 | "Inference tables require manual setup" | They are automatically created when you enable them on a serving endpoint |
| 2 | "Inference tables only store inputs and outputs" | They also store metadata: timestamp, latency, token counts, request ID |
| 3 | "Monitoring requires external tools" | Databricks provides built-in monitoring via inference tables + SQL queries |
| 4 | "Inference tables are the same as MLflow experiment logs" | Inference tables = production traffic logs; experiments = offline evaluation |

---

## Prerequisites

- Lab 06-01 completed (familiarity with evaluation concepts)
- Understanding of Model Serving endpoints (Module 04)

---

## Exam Objectives Covered

- Enabling inference tables on serving endpoints
- Inference table schema: what data is captured
- Querying inference tables with SQL for monitoring
- Building monitoring dashboards and alerts
- Connecting inference table data back to MLflow evaluation
- Offline evaluation vs online monitoring

## Setup

In [ ]:
%pip install mlflow>=2.14 openai databricks-sdk -q
dbutils.library.restartPython()

In [ ]:
import mlflow
import json
import pandas as pd
from datetime import datetime, timedelta
from openai import OpenAI
import random

# Databricks Foundation Model API via OpenAI-compatible endpoint
client = OpenAI(
    api_key=dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get(),
    base_url=f"https://{spark.conf.get('spark.databricks.workspaceUrl')}/serving-endpoints"
)

MODEL = "databricks-meta-llama-3-3-70b-instruct"
CATALOG = "workspace"
SCHEMA = "genai_labs"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
print(f"Model:   {MODEL}")
print(f"Catalog: {CATALOG}")
print(f"Schema:  {SCHEMA}")
print(f"MLflow:  {mlflow.__version__}")

Expected output:
```
Model:   databricks-meta-llama-3-3-70b-instruct
Catalog: main
Schema:  genai_labs
MLflow:  2.14.x
```

---

## Step 1: Enabling Inference Tables (Endpoint Config, Storage Location)

Inference tables are enabled per-endpoint in the Serving UI or via API. When enabled,
Databricks **automatically** creates a Delta table that logs every request and response.

**UI ->** Left nav -> **Serving** -> Select endpoint -> **Edit configuration** -> **Enable inference tables**

```
User Request ---> Serving Endpoint ---> Response to User
                       |
                       +---> Inference Table (Delta)
                              - request payload
                              - response payload
                              - timestamp
                              - execution_time_ms
                              - token counts
                              - request_id
                              - status_code
```

> **Key point:** You do NOT build the logging pipeline. You just toggle a switch. Databricks
> handles the rest -- schema creation, writing, partitioning, and retention.

In [ ]:
# In production, you enable inference tables via the API:
# This is a reference example -- do not run against a live endpoint without proper setup

enable_inference_table_config = {
    "name": "my-rag-endpoint",
    "config": {
        "served_entities": [
            {
                "entity_name": "my-rag-model",
                "entity_version": "1",
                "scale_to_zero_enabled": True,
                "workload_size": "Small",
            }
        ],
        # THIS IS THE KEY SETTING -- auto_capture_config:
        "auto_capture_config": {
            "catalog_name": CATALOG,
            "schema_name": SCHEMA,
            "table_name_prefix": "my_rag_endpoint",
            "enabled": True,
        }
    }
}

print("Inference table configuration:")
print(json.dumps(enable_inference_table_config["config"]["auto_capture_config"], indent=2))
print()
print("When enabled, Databricks creates:")
print(f"  Payload table:  {CATALOG}.{SCHEMA}.my_rag_endpoint_payload")
print(f"  Contains:       every request/response pair with metadata")
print(f"  Format:         Delta table (queryable via SQL)")
print(f"  Auto-managed:   schema, writes, partitioning, retention")

Expected output:
```
Inference table configuration:
{
  "catalog_name": "workspace",
  "schema_name": "genai_labs",
  "table_name_prefix": "my_rag_endpoint",
  "enabled": true
}

When enabled, Databricks creates:
  Payload table:  workspace.genai_labs.my_rag_endpoint_payload
  Contains:       every request/response pair with metadata
  Format:         Delta table (queryable via SQL)
  Auto-managed:   schema, writes, partitioning, retention
```

> **Exam trap:** The exam may say "Inference tables require manual setup" -- this is **false**.
> They are automatically created when you enable `auto_capture_config` on the endpoint. No
> schema definition, no pipeline code, no ETL. Just toggle the config.

---

## Step 2: Schema Deep Dive (Columns, Types, Example Rows)

Understanding the schema is critical for querying and monitoring. Every row in the inference
table represents a single request/response pair.

| Column | Type | Description |
|--------|------|-------------|
| `databricks_request_id` | STRING | Unique identifier for each request |
| `date` | DATE | Partition column for efficient queries |
| `timestamp_ms` | LONG | Unix timestamp in milliseconds |
| `status_code` | INT | HTTP status code (200 = success, 4xx/5xx = error) |
| `execution_time_ms` | LONG | End-to-end latency in milliseconds |
| `request` | STRING | Full JSON request payload (messages, parameters) |
| `response` | STRING | Full JSON response payload (choices, usage) |
| `request_metadata` | MAP<STRING,STRING> | Endpoint name, model name, user info |
| `sampling_fraction` | DOUBLE | Fraction of requests logged (usually 1.0) |
| `client_request_id` | STRING | Client-specified request ID (if provided) |

> **Key insight for exam:** The inference table stores BOTH the full request AND response
> payloads as JSON strings. To extract specific fields (like the user's question or the
> token count), you use `get_json_object()` or `from_json()` in SQL.

Let's create a realistic simulated inference table to practice querying.

In [ ]:
# Create a simulated inference table with realistic data
# In production, this table is auto-populated by the serving endpoint

random.seed(42)
base_time = datetime(2025, 1, 15, 9, 0, 0)

questions = [
    "What is the return policy for the SmartHome Hub?",
    "How do I connect a new device to the Hub?",
    "What voice assistants are supported?",
    "My Hub is offline, what should I do?",
    "What encryption does the Hub use?",
    "How do I set up automation rules?",
    "What warranty does the Hub come with?",
    "How do I enable voice control?",
    "The Hub LED is flashing red, what does it mean?",
    "Can I use the Hub without internet?",
]

answers = [
    "Returns are accepted within 30 days for a full refund, provided the product is in original packaging.",
    "Go to the Devices tab in the Acme Home app, tap plus, select device type, and follow pairing prompts.",
    "The Hub supports Amazon Alexa, Google Assistant, and Apple Siri.",
    "Check your internet connection, verify cable connections, and power cycle the Hub for 30 seconds.",
    "All device communication uses AES-256 encryption.",
    "Navigate to Automations tab, create triggers and conditions, then assign actions to devices.",
    "The Hub includes a 2-year limited warranty covering manufacturing defects.",
    "Open the Acme Home app, go to Settings > Integrations, and select your voice assistant.",
    "A flashing red LED indicates a hardware fault. Contact Acme Support immediately.",
    "The Hub requires internet for cloud features, but local automations work offline.",
]

rows = []
for i in range(200):  # 200 requests over the day
    q_idx = random.randint(0, len(questions) - 1)
    status = 200 if random.random() > 0.05 else random.choice([400, 429, 500, 503])
    latency = random.gauss(350, 120)
    latency = max(50, min(3000, latency))  # clamp to realistic range

    # Inject latency spikes in hours 12-13 (lunch rush)
    request_time = base_time + timedelta(minutes=random.randint(0, 540))
    if 12 <= request_time.hour <= 13:
        latency *= 1.5

    prompt_tokens = random.randint(80, 250)
    completion_tokens = random.randint(30, 180) if status == 200 else 0
    total_tokens = prompt_tokens + completion_tokens

    request_payload = json.dumps({
        "messages": [
            {"role": "system", "content": "You are a helpful product support assistant."},
            {"role": "user", "content": questions[q_idx]}
        ],
        "max_tokens": 300,
        "temperature": 0.0,
    })

    response_payload = json.dumps({
        "id": f"chatcmpl-{i:06d}",
        "choices": [{"index": 0, "message": {"role": "assistant",
            "content": answers[q_idx] if status == 200 else ""},
            "finish_reason": "stop" if status == 200 else "error"}],
        "usage": {
            "prompt_tokens": prompt_tokens,
            "completion_tokens": completion_tokens,
            "total_tokens": total_tokens,
        }
    })

    rows.append({
        "databricks_request_id": f"req-{i:04d}",
        "date": request_time.strftime("%Y-%m-%d"),
        "timestamp_ms": int(request_time.timestamp() * 1000),
        "timestamp": request_time.isoformat(),
        "status_code": status,
        "execution_time_ms": int(latency),
        "request": request_payload,
        "response": response_payload,
        "prompt_tokens": prompt_tokens,
        "completion_tokens": completion_tokens,
        "total_tokens": total_tokens,
        "endpoint_name": "my-rag-endpoint",
    })

# Save as a Delta table
inference_df = spark.createDataFrame(rows)
table_name = f"{CATALOG}.{SCHEMA}.inference_table_demo"
inference_df.write.mode("overwrite").saveAsTable(table_name)

print(f"Created simulated inference table: {table_name}")
print(f"Total requests:  {inference_df.count()}")
print(f"Columns:         {len(inference_df.columns)}")
print()
print("Sample rows:")
spark.sql(f"""
    SELECT databricks_request_id, timestamp, status_code, execution_time_ms
    FROM {table_name}
    ORDER BY timestamp
    LIMIT 5
""").show(truncate=False)

Expected output:
```
Created simulated inference table: workspace.genai_labs.inference_table_demo
Total requests:  200
Columns:         12

Sample rows:
+----------------------+-------------------+-----------+------------------+
|databricks_request_id |timestamp          |status_code|execution_time_ms |
+----------------------+-------------------+-----------+------------------+
|req-0012              |2025-01-15T09:02:00|200        |312               |
|req-0045              |2025-01-15T09:05:00|200        |278               |
|req-0078              |2025-01-15T09:08:00|500        |445               |
|req-0023              |2025-01-15T09:12:00|200        |389               |
|req-0091              |2025-01-15T09:15:00|200        |301               |
+----------------------+-------------------+-----------+------------------+
```

---

## Step 3: SQL Queries for Latency Analysis and Error Rates

The first monitoring step: query the inference table for overall endpoint health.
These are the queries you would run in the SQL Editor or embed in a dashboard.

**UI ->** Left nav -> **SQL Editor** -> Write and run queries against the inference table

In [ ]:
# Query 1: Overall endpoint health summary
summary = spark.sql(f"""
SELECT
    COUNT(*)                                                           AS total_requests,
    SUM(CASE WHEN status_code = 200 THEN 1 ELSE 0 END)               AS successful,
    SUM(CASE WHEN status_code != 200 THEN 1 ELSE 0 END)              AS failed,
    ROUND(SUM(CASE WHEN status_code != 200 THEN 1 ELSE 0 END)
          * 100.0 / COUNT(*), 1)                                      AS error_rate_pct,
    ROUND(AVG(execution_time_ms), 0)                                  AS avg_latency_ms,
    ROUND(PERCENTILE(execution_time_ms, 0.50), 0)                     AS p50_latency_ms,
    ROUND(PERCENTILE(execution_time_ms, 0.95), 0)                     AS p95_latency_ms,
    ROUND(PERCENTILE(execution_time_ms, 0.99), 0)                     AS p99_latency_ms
FROM {CATALOG}.{SCHEMA}.inference_table_demo
""")

print("=" * 65)
print("ENDPOINT HEALTH SUMMARY")
print("=" * 65)
summary.show(truncate=False)

Expected output:
```
=================================================================
ENDPOINT HEALTH SUMMARY
=================================================================
+--------------+----------+------+--------------+--------------+--------------+--------------+--------------+
|total_requests|successful|failed|error_rate_pct|avg_latency_ms|p50_latency_ms|p95_latency_ms|p99_latency_ms|
+--------------+----------+------+--------------+--------------+--------------+--------------+--------------+
|200           |190       |10    |5.0           |365           |348           |545           |680           |
+--------------+----------+------+--------------+--------------+--------------+--------------+--------------+
```

> **Reading the metrics:**
> - **error_rate_pct = 5.0**: 5% of requests failed -- investigate status codes
> - **p50 = 348ms**: half of requests complete within 348ms (median user experience)
> - **p95 = 545ms**: 95% of requests complete within 545ms (tail latency for SLA)
> - **p99 = 680ms**: the slowest 1% -- watch for outliers that hurt user experience

In [ ]:
# Query 2: Token usage and cost estimation
# Typical pricing: ~$0.001 per 1K input tokens, ~$0.002 per 1K output tokens
token_usage = spark.sql(f"""
SELECT
    SUM(prompt_tokens)                                     AS total_input_tokens,
    SUM(completion_tokens)                                 AS total_output_tokens,
    SUM(total_tokens)                                      AS total_all_tokens,
    ROUND(AVG(prompt_tokens), 0)                           AS avg_input_per_req,
    ROUND(AVG(completion_tokens), 0)                       AS avg_output_per_req,
    ROUND(SUM(prompt_tokens) * 0.001 / 1000
        + SUM(completion_tokens) * 0.002 / 1000, 4)       AS est_cost_usd
FROM {CATALOG}.{SCHEMA}.inference_table_demo
WHERE status_code = 200
""")

print("TOKEN USAGE & COST ESTIMATE")
print("=" * 65)
token_usage.show(truncate=False)

Expected output:
```
TOKEN USAGE & COST ESTIMATE
=================================================================
+------------------+-------------------+---------------+----------------+-----------------+------------+
|total_input_tokens|total_output_tokens|total_all_tokens|avg_input_per_req|avg_output_per_req|est_cost_usd|
+------------------+-------------------+---------------+----------------+-----------------+------------+
|28500             |17800              |46300          |150             |94               |0.0641      |
+------------------+-------------------+---------------+----------------+-----------------+------------+
```

> **Why this matters:** Token usage directly drives cost. Tracking average tokens per request
> helps you estimate monthly spend and identify prompts that are too verbose.

In [ ]:
# Query 3: Error breakdown by status code
errors = spark.sql(f"""
SELECT
    status_code,
    COUNT(*) AS count,
    ROUND(AVG(execution_time_ms), 0) AS avg_latency_ms,
    CASE
        WHEN status_code = 200 THEN 'Success'
        WHEN status_code = 400 THEN 'Bad request (malformed input)'
        WHEN status_code = 429 THEN 'Rate limited (too many requests)'
        WHEN status_code = 500 THEN 'Internal server error'
        WHEN status_code = 503 THEN 'Service unavailable (scaling/overloaded)'
        ELSE 'Other'
    END AS meaning
FROM {CATALOG}.{SCHEMA}.inference_table_demo
GROUP BY status_code
ORDER BY count DESC
""")

print("ERROR BREAKDOWN BY STATUS CODE")
print("=" * 65)
errors.show(truncate=False)

Expected output:
```
ERROR BREAKDOWN BY STATUS CODE
=================================================================
+-----------+-----+--------------+-----------------------------------+
|status_code|count|avg_latency_ms|meaning                            |
+-----------+-----+--------------+-----------------------------------+
|200        |190  |355           |Success                            |
|429        |4    |120           |Rate limited (too many requests)   |
|500        |3    |445           |Internal server error              |
|503        |2    |80            |Service unavailable (scaling)      |
|400        |1    |95            |Bad request (malformed input)      |
+-----------+-----+--------------+-----------------------------------+
```

> **Actionable insight:** 429 errors (rate limiting) are the most common failure. This tells
> you either traffic is exceeding limits or the rate limit config needs adjustment.
> See Lab 06-04 (AI Gateway) for rate limit management.

---

## Step 4: Building Monitoring Dashboards (SQL Queries for Key Metrics)

Monitoring is about spotting **trends**, not just point-in-time snapshots. Hourly aggregations
reveal traffic patterns, latency spikes, and error rate changes over time.

**UI ->** Left nav -> **SQL Editor** -> Create queries -> **Dashboards** -> Add visualizations

A production monitoring dashboard typically has 4-5 panels, each powered by a SQL query.

In [ ]:
# Dashboard Panel 1: Hourly traffic and latency trends
hourly_trends = spark.sql(f"""
SELECT
    HOUR(timestamp) AS hour,
    COUNT(*) AS requests,
    SUM(CASE WHEN status_code != 200 THEN 1 ELSE 0 END) AS errors,
    ROUND(SUM(CASE WHEN status_code != 200 THEN 1 ELSE 0 END)
          * 100.0 / COUNT(*), 1) AS error_rate_pct,
    ROUND(AVG(execution_time_ms), 0) AS avg_latency_ms,
    ROUND(PERCENTILE(execution_time_ms, 0.95), 0) AS p95_latency_ms,
    SUM(total_tokens) AS total_tokens
FROM {CATALOG}.{SCHEMA}.inference_table_demo
GROUP BY HOUR(timestamp)
ORDER BY hour
""")

print("HOURLY TRAFFIC & LATENCY TRENDS")
print("=" * 75)
hourly_trends.show(20, truncate=False)

Expected output:
```
HOURLY TRAFFIC & LATENCY TRENDS
===========================================================================
+----+--------+------+--------------+--------------+--------------+------------+
|hour|requests|errors|error_rate_pct|avg_latency_ms|p95_latency_ms|total_tokens|
+----+--------+------+--------------+--------------+--------------+------------+
|9   |22      |1     |4.5           |345           |490           |5060        |
|10  |25      |1     |4.0           |330           |478           |5750        |
|11  |24      |2     |8.3           |360           |510           |5520        |
|12  |28      |2     |7.1           |475           |640           |6440        |
|13  |26      |1     |3.8           |460           |620           |5980        |
|14  |22      |1     |4.5           |355           |520           |5060        |
|15  |20      |1     |5.0           |340           |485           |4600        |
|16  |18      |1     |5.6           |348           |495           |4140        |
|17  |15      |0     |0.0           |335           |470           |3450        |
+----+--------+------+--------------+--------------+--------------+------------+
```

> **What to look for:**
> - **Hours 12-13** show higher latency (475ms, 460ms vs ~340ms baseline) -- this is the
>   "lunch rush" traffic spike causing slower responses
> - **Error rate** stays relatively stable -- errors are not correlated with traffic volume
> - **Token usage** peaks with traffic -- expected, but watch for sudden increases (prompt bloat)

In [ ]:
# Dashboard Panel 2: Most common questions (extract from request JSON)
common_questions = spark.sql(f"""
SELECT
    get_json_object(request, '$.messages[1].content') AS question,
    COUNT(*) AS frequency,
    ROUND(AVG(execution_time_ms), 0) AS avg_latency_ms,
    ROUND(AVG(total_tokens), 0) AS avg_tokens
FROM {CATALOG}.{SCHEMA}.inference_table_demo
WHERE status_code = 200
GROUP BY get_json_object(request, '$.messages[1].content')
ORDER BY frequency DESC
LIMIT 10
""")

print("MOST FREQUENT QUESTIONS")
print("=" * 75)
common_questions.show(truncate=55)

Expected output:
```
MOST FREQUENT QUESTIONS
===========================================================================
+-------------------------------------------------------+---------+--------------+----------+
|question                                               |frequency|avg_latency_ms|avg_tokens|
+-------------------------------------------------------+---------+--------------+----------+
|What is the return policy for the SmartHome Hub?       |22       |340           |210       |
|How do I connect a new device to the Hub?              |21       |355           |225       |
|My Hub is offline, what should I do?                   |20       |380           |240       |
|What voice assistants are supported?                   |19       |320           |195       |
+-------------------------------------------------------+---------+--------------+----------+
```

> **Why this matters:** Knowing the most common questions helps you:
> 1. Prioritize your evaluation dataset (test the questions users actually ask)
> 2. Identify knowledge gaps (new question types = missing documentation)
> 3. Optimize prompts for high-frequency queries

In [ ]:
# Dashboard Panel 3: Latency distribution (histogram buckets)
latency_dist = spark.sql(f"""
SELECT
    CASE
        WHEN execution_time_ms < 200 THEN '< 200ms'
        WHEN execution_time_ms < 400 THEN '200-400ms'
        WHEN execution_time_ms < 600 THEN '400-600ms'
        WHEN execution_time_ms < 1000 THEN '600ms-1s'
        ELSE '> 1s'
    END AS latency_bucket,
    COUNT(*) AS count,
    ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM {CATALOG}.{SCHEMA}.inference_table_demo), 1) AS pct
FROM {CATALOG}.{SCHEMA}.inference_table_demo
GROUP BY 1
ORDER BY MIN(execution_time_ms)
""")

print("LATENCY DISTRIBUTION")
print("=" * 50)
latency_dist.show(truncate=False)

Expected output:
```
LATENCY DISTRIBUTION
==================================================
+--------------+-----+----+
|latency_bucket|count|pct |
+--------------+-----+----+
|< 200ms       |18   |9.0 |
|200-400ms     |112  |56.0|
|400-600ms     |52   |26.0|
|600ms-1s      |15   |7.5 |
|> 1s          |3    |1.5 |
+--------------+-----+----+
```

> **SLA check:** If your SLA says "95% of requests under 600ms", the data shows 91%
> under 600ms -- you are below target. The 400-600ms bucket is where to focus optimization.

---

## Step 5: Alerting on Anomalies (Quality Drops, Latency Spikes)

Dashboards show you what happened. **Alerts** tell you the moment something goes wrong.
Databricks SQL Alerts run a query on a schedule and trigger notifications when a threshold
is crossed.

**UI ->** Left nav -> **SQL Editor** -> Create query -> **Alerts** -> Set threshold

```
+-------------------+     +------------------+     +------------------+
|  SQL Query runs   | --> | Result crosses   | --> | Notification     |
|  every 5 minutes  |     | threshold?       |     | Email / Slack /  |
|                   |     | error_rate > 5%  |     | PagerDuty        |
+-------------------+     +------------------+     +------------------+
```

In [ ]:
# Alert definitions -- in production, configure these in the SQL Alerts UI

alert_configs = [
    {
        "name": "High Error Rate",
        "schedule": "Every 5 minutes",
        "query": f"""
            SELECT ROUND(
                SUM(CASE WHEN status_code != 200 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1
            ) AS error_rate_pct
            FROM {CATALOG}.{SCHEMA}.inference_table_demo
            WHERE timestamp >= current_timestamp() - INTERVAL 1 HOUR
        """,
        "condition": "error_rate_pct > 5.0",
        "action": "Email + Slack notification",
        "why": "Catch endpoint failures before users report them",
    },
    {
        "name": "Latency Spike (p95)",
        "schedule": "Every 5 minutes",
        "query": f"""
            SELECT ROUND(PERCENTILE(execution_time_ms, 0.95), 0) AS p95_latency
            FROM {CATALOG}.{SCHEMA}.inference_table_demo
            WHERE timestamp >= current_timestamp() - INTERVAL 1 HOUR
        """,
        "condition": "p95_latency > 1000",
        "action": "Email + PagerDuty",
        "why": "Latency spikes degrade user experience and may indicate scaling issues",
    },
    {
        "name": "Traffic Drop (Zero Traffic)",
        "schedule": "Every 15 minutes",
        "query": f"""
            SELECT COUNT(*) AS request_count
            FROM {CATALOG}.{SCHEMA}.inference_table_demo
            WHERE timestamp >= current_timestamp() - INTERVAL 1 HOUR
        """,
        "condition": "request_count < 5",
        "action": "Email notification",
        "why": "Sudden traffic drop may mean endpoint is down or routing changed",
    },
    {
        "name": "Cost Spike (Token Usage)",
        "schedule": "Every 1 hour",
        "query": f"""
            SELECT SUM(total_tokens) AS hourly_tokens
            FROM {CATALOG}.{SCHEMA}.inference_table_demo
            WHERE timestamp >= current_timestamp() - INTERVAL 1 HOUR
        """,
        "condition": "hourly_tokens > 100000",
        "action": "Email to team lead",
        "why": "Unexpected token usage spike may indicate prompt injection or abuse",
    },
]

print("ALERT CONFIGURATIONS")
print("=" * 65)
for alert in alert_configs:
    print(f"\nAlert: {alert['name']}")
    print(f"  Schedule: {alert['schedule']}")
    print(f"  Trigger:  {alert['condition']}")
    print(f"  Action:   {alert['action']}")
    print(f"  Why:      {alert['why']}")

Expected output:
```
ALERT CONFIGURATIONS
=================================================================

Alert: High Error Rate
  Schedule: Every 5 minutes
  Trigger:  error_rate_pct > 5.0
  Action:   Email + Slack notification
  Why:      Catch endpoint failures before users report them

Alert: Latency Spike (p95)
  Schedule: Every 5 minutes
  Trigger:  p95_latency > 1000
  Action:   Email + PagerDuty
  Why:      Latency spikes degrade user experience and may indicate scaling issues

Alert: Traffic Drop (Zero Traffic)
  Schedule: Every 15 minutes
  Trigger:  request_count < 5
  Action:   Email notification
  Why:      Sudden traffic drop may mean endpoint is down or routing changed

Alert: Cost Spike (Token Usage)
  Schedule: Every 1 hour
  Trigger:  hourly_tokens > 100000
  Action:   Email to team lead
  Why:      Unexpected token usage spike may indicate prompt injection or abuse
```

> **Exam note:** You do NOT need to know the exact SQL Alert API syntax. Know that alerts
> are SQL-query-based, run on a schedule, and trigger notifications when thresholds are crossed.

In [ ]:
# Simulate checking alert conditions against our data
print("ALERT STATUS CHECK (simulated)")
print("=" * 65)

# Check error rate
error_check = spark.sql(f"""
    SELECT ROUND(
        SUM(CASE WHEN status_code != 200 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1
    ) AS error_rate_pct
    FROM {CATALOG}.{SCHEMA}.inference_table_demo
""").collect()[0]
error_rate = error_check["error_rate_pct"]
print(f"\nHigh Error Rate: {error_rate}%  {'TRIGGERED' if error_rate > 5.0 else 'OK'}")

# Check p95 latency
latency_check = spark.sql(f"""
    SELECT ROUND(PERCENTILE(execution_time_ms, 0.95), 0) AS p95_latency
    FROM {CATALOG}.{SCHEMA}.inference_table_demo
""").collect()[0]
p95 = latency_check["p95_latency"]
print(f"Latency Spike:   p95={p95}ms  {'TRIGGERED' if p95 > 1000 else 'OK'}")

# Check traffic
traffic_check = spark.sql(f"""
    SELECT COUNT(*) AS request_count
    FROM {CATALOG}.{SCHEMA}.inference_table_demo
""").collect()[0]
traffic = traffic_check["request_count"]
print(f"Traffic Drop:    {traffic} requests  {'TRIGGERED' if traffic < 5 else 'OK'}")

# Check token usage
token_check = spark.sql(f"""
    SELECT SUM(total_tokens) AS total_tokens
    FROM {CATALOG}.{SCHEMA}.inference_table_demo
""").collect()[0]
tokens = token_check["total_tokens"]
print(f"Cost Spike:      {tokens} tokens  {'TRIGGERED' if tokens > 100000 else 'OK'}")

Expected output:
```
ALERT STATUS CHECK (simulated)
=================================================================

High Error Rate: 5.0%  OK
Latency Spike:   p95=545ms  OK
Traffic Drop:    200 requests  OK
Cost Spike:      46300 tokens  OK
```

---

## Step 6: Connecting Inference Data to MLflow Evaluate

The power move: sample real production requests from inference tables, then run them through
`mlflow.genai.evaluate()` for quality scoring. This bridges online monitoring with offline
evaluation.

```
Production Traffic -> Inference Table -> Sample N requests -> mlflow.genai.evaluate()
                                                                      |
                                                              Quality scores
                                                              (per-request)
                                                                      |
                                                              Dashboard + Alerts
```

**Why?** Inference tables give you latency and error rates, but NOT quality metrics
(faithfulness, relevance). By sampling and evaluating, you get the full picture.

In [ ]:
# Step 1: Sample production requests from the inference table
sampled = spark.sql(f"""
SELECT
    get_json_object(request, '$.messages[1].content') AS question,
    get_json_object(response, '$.choices[0].message.content') AS answer,
    execution_time_ms,
    total_tokens,
    databricks_request_id
FROM {CATALOG}.{SCHEMA}.inference_table_demo
WHERE status_code = 200
ORDER BY RAND(42)
LIMIT 10
""").toPandas()

print(f"Sampled {len(sampled)} production requests for quality evaluation")
print()

# Step 2: Convert to mlflow.genai.evaluate() format
eval_from_production = []
for _, row in sampled.iterrows():
    eval_from_production.append({
        "inputs": {"question": row["question"]},
        # Note: no expected_response -- we use reference-free scorers
        # Or we could add SME-provided ground truth (see Lab 06-05)
    })

print("Converted to evaluation format:")
for i, ex in enumerate(eval_from_production[:3]):
    print(f"  [{i}] {ex['inputs']['question'][:60]}...")
print(f"  ... ({len(eval_from_production)} total)")
print()
print("Available scorer types for production data:")
print("  WITH ground truth:    Faithfulness, Relevance (need expected_response)")
print("  WITHOUT ground truth: Safety, Guidelines (reference-free)")
print("  See Lab 06-05 for collecting SME ground truth")

Expected output:
```
Sampled 10 production requests for quality evaluation

Converted to evaluation format:
  [0] What is the return policy for the SmartHome Hub?...
  [1] My Hub is offline, what should I do?...
  [2] How do I enable voice control?...
  ... (10 total)

Available scorer types for production data:
  WITH ground truth:    Faithfulness, Relevance (need expected_response)
  WITHOUT ground truth: Safety, Guidelines (reference-free)
  See Lab 06-05 for collecting SME ground truth
```

In [ ]:
# Step 3: Run reference-free evaluation on production samples
from mlflow.genai.scorers import Safety, Guidelines

# Define reference-free scorers (do not need expected_response)
production_scorers = [
    Safety(),
    Guidelines(
        name="professional_tone",
        definition="The response uses professional, courteous language appropriate for customer support."
    ),
    Guidelines(
        name="actionable",
        definition="The response provides specific, actionable information rather than vague suggestions."
    ),
]

# For pre-recorded outputs, we include the actual production response in the data
production_eval_data = []
for _, row in sampled.iterrows():
    production_eval_data.append({
        "inputs": {"question": row["question"]},
        "outputs": row["answer"],  # Use the actual production response
    })

mlflow.set_experiment("/Users/you/genai-labs/06-03-inference-monitoring")

# When outputs are pre-provided in data, predict_fn is not needed
eval_results = mlflow.genai.evaluate(
    data=production_eval_data,
    scorers=production_scorers,
)

print("Production quality evaluation complete!")
print()
print("Aggregate metrics:")
for metric, value in sorted(eval_results.metrics.items()):
    if isinstance(value, float):
        print(f"  {metric:<35} {value:.3f}")

Expected output:
```
Production quality evaluation complete!

Aggregate metrics:
  actionable/mean                       0.900
  professional_tone/mean                1.000
  safety/mean                           1.000
```

> **The complete picture:** Now you have BOTH operational metrics (latency, errors, tokens)
> from SQL queries AND quality metrics (safety, tone, actionability) from MLflow evaluate.
> This is comprehensive production monitoring.

---

## Offline Evaluation vs Online Monitoring

A critical distinction for the exam:

| Dimension | Offline Evaluation | Online Monitoring |
|-----------|-------------------|------------------|
| **When** | Before deployment | After deployment |
| **Data source** | Curated eval dataset | Real production traffic (inference tables) |
| **Tool** | `mlflow.genai.evaluate()` | Inference tables + SQL queries |
| **Ground truth** | Expected responses provided | No ground truth (usually) |
| **Scorers** | All scorers (faithfulness, relevance, etc.) | Reference-free, OR add SME feedback |
| **Frequency** | On-demand or in CI/CD | Continuous |
| **Purpose** | Validate before shipping | Detect regressions in production |
| **Latency/errors** | Not measured | Primary focus |
| **Quality scores** | Primary focus | Requires sampling + evaluate() |

> **Exam key point:** Both are needed. Offline evaluation gates deployment. Online monitoring
> catches regressions. They complement each other -- one is not a replacement for the other.

In [ ]:
# The complete monitoring workflow -- putting it all together
print("=" * 65)
print("THE COMPLETE EVALUATION + MONITORING STACK")
print("=" * 65)
print()
print("PRE-DEPLOYMENT (Offline Evaluation):")
print("  1. Create eval dataset with expected responses")
print("  2. Run mlflow.genai.evaluate() with all scorers")
print("  3. Compare configs in Experiments UI")
print("  4. Gate deployment on quality thresholds")
print()
print("POST-DEPLOYMENT (Online Monitoring):")
print("  5. Enable inference tables on serving endpoint")
print("  6. Query for latency (p50/p95/p99), error rates, token usage")
print("  7. Build monitoring dashboard in SQL Editor")
print("  8. Configure SQL Alerts for anomaly detection")
print("  9. Sample production requests for periodic quality eval")
print()
print("CONTINUOUS IMPROVEMENT:")
print("  10. Analyze most common questions from inference tables")
print("  11. Add high-frequency questions to eval dataset")
print("  12. Collect SME feedback on production responses (Lab 06-05)")
print("  13. Use feedback as ground truth for re-evaluation")
print("  14. Deploy improved version and repeat")
print()
print("     +----------+     +---------+     +----------+")
print("     |  Evaluate | --> | Deploy  | --> | Monitor  |")
print("     +----------+     +---------+     +----------+")
print("          ^                                 |")
print("          |    +----------+    +----------+ |")
print("          +--- | Improve  | <- | Feedback |-+")
print("               +----------+    +----------+")

---

## Where Inference Tables Live in Unity Catalog

Inference tables are stored as regular Delta tables in Unity Catalog:

**UI ->** Left nav -> **Catalog** -> Navigate to your catalog/schema -> Find the payload table

```
Unity Catalog
  +-- main (catalog)
       +-- genai_labs (schema)
            +-- my_rag_endpoint_payload          (inference table)
            +-- my_rag_endpoint_payload_assessment (assessment table, if Review App)
```

> **Exam note:** Inference tables are governed by Unity Catalog permissions. Only users
> with SELECT access on the table can query the logs. This is important for compliance
> and auditing -- production request/response data may contain sensitive information.

In [ ]:
# Summary: what we covered
print("=" * 65)
print("Lab 06-03 Summary")
print("=" * 65)
print()
print("1. ENABLING INFERENCE TABLES")
print("   - auto_capture_config on serving endpoint")
print("   - Automatically creates Delta table in Unity Catalog")
print("   - No manual schema or pipeline setup needed")
print()
print("2. INFERENCE TABLE SCHEMA")
print("   - request_id, timestamp, status_code, execution_time_ms")
print("   - request (full JSON), response (full JSON)")
print("   - Token counts: prompt_tokens, completion_tokens, total_tokens")
print()
print("3. SQL MONITORING QUERIES")
print("   - Overall health: error rate, latency percentiles (p50/p95/p99)")
print("   - Token usage and cost estimation")
print("   - Hourly trends for spotting patterns")
print("   - Most common questions for eval dataset prioritization")
print()
print("4. ALERTING")
print("   - SQL Alerts: query + threshold + schedule + notification")
print("   - Key alerts: error rate, latency spike, traffic drop, cost spike")
print()
print("5. CONNECTING TO MLFLOW EVALUATE")
print("   - Sample production requests from inference tables")
print("   - Run through mlflow.genai.evaluate() with reference-free scorers")
print("   - Bridges operational metrics with quality metrics")

---

## Exam Tips

| # | Tip | Why it matters |
|---|-----|----------------|
| 1 | Inference tables are auto-created Delta tables that log every request/response | Know the automatic nature -- no manual setup |
| 2 | Enable via `auto_capture_config` on the serving endpoint | Know the configuration path |
| 3 | Key columns: request_id, timestamp, status_code, execution_time_ms, request, response | Know what data is captured |
| 4 | Use SQL queries for latency (p50/p95/p99), error rates, token usage | Know the key monitoring queries |
| 5 | SQL Alerts trigger notifications when metrics cross thresholds | Know the alerting mechanism |
| 6 | Offline evaluation uses curated data; online monitoring uses production traffic | Critical distinction for the exam |
| 7 | Sample production requests + mlflow.evaluate() = quality monitoring | Know how to bridge the two |

---

## Key Takeaways

**Inference Tables**
- Automatically created Delta tables that log every serving endpoint request/response
- Capture: request payload, response payload, timestamp, latency, token counts, status code
- Stored in Unity Catalog -- governed by standard permissions
- Queryable via SQL Editor -- no external tools needed

**Monitoring Queries**
- Health: error rate, latency percentiles (p50/p95/p99), request volume
- Cost: token usage tracking, cost estimation per request and over time
- Patterns: hourly trends, most common questions, error breakdown by status code
- Use `get_json_object()` to extract fields from request/response JSON

**Alerting**
- SQL Alerts: SQL query + threshold + schedule + notification channel
- Key alerts: error rate > threshold, p95 latency spike, traffic drop, cost spike
- Runs automatically on schedule -- no manual monitoring needed

**Bridging Evaluation and Monitoring**
- Sample production requests from inference tables
- Run through `mlflow.genai.evaluate()` for quality scoring
- Without ground truth: use reference-free scorers (Safety, Guidelines)
- With SME feedback (Lab 06-05): use all scorers including Faithfulness and Relevance

---

**Next Lab:** [06-04: AI Gateway](./04_ai_gateway.ipynb) -- Unified proxy layer for all LLM endpoints with rate limiting, usage tracking, and cost management.